# BuildCompiler Offline Quickstart

This notebook shows how to use BuildCompiler without SynBioHub network access. It loads local SBOL collections, joins them into one working SBOL document, indexes plasmids/backbones/reagents, and runs a level-1 MoClo assembly from a local design fixture.

## 1. Imports and Repository Paths

Run this notebook from the repository checkout after installing BuildCompiler with `python -m pip install -e .`.

In [ ]:
import json
from pathlib import Path

import sbol2

from buildcompiler.api import BuildCompiler as OrchestratedBuildCompiler
from buildcompiler.api import BuildOptions, domestication as run_domestication
from buildcompiler.buildcompiler import BuildCompiler
from buildcompiler.abstract_translator import extract_toplevel_definition
from buildcompiler.adapters.pudu import domestication_artifacts_to_pudu_json, write_assembly_pudu_input_json
from buildcompiler.domain import BuildRequest, BuildStage, DesignKind, IndexedBackbone, IndexedPlasmid, IndexedReagent, MaterialState, StageStatus
from buildcompiler.execution import FullBuildExecutor
from buildcompiler.inventory import Inventory
from buildcompiler.planning import BuildPlan
from buildcompiler.sbol import AssemblySbolResult
from buildcompiler.stages import AssemblyLvl1Stage


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "tests" / "test_files").exists():
            return path
    raise RuntimeError("Could not find BuildCompiler repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
TEST_FILES = REPO_ROOT / "tests" / "test_files"
RESULTS_DIR = REPO_ROOT / "notebooks" / "results" / "buildcompiler_offline_quickstart"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Results:", RESULTS_DIR)

## 2. Load Local SBOL Collections

These four files replace the SynBioHub collection pulls used in older examples. BuildCompiler merges them into one local SBOL document and indexes the material inventory from that merged document.

In [ ]:
collection_paths = [
    TEST_FILES / "CIDARMoCloParts_collection.xml",
    TEST_FILES / "CIDARMoCloPlasmidsKit_collection.xml",
    TEST_FILES / "Enzyme_Implementations_collection.xml",
    TEST_FILES / "impl_test_collection.xml",
]

collection_docs = []
for path in collection_paths:
    doc = sbol2.Document()
    doc.read(str(path))
    collection_docs.append(doc)
    print(path.name, "components=", len(doc.componentDefinitions), "modules=", len(doc.moduleDefinitions), "implementations=", len(doc.implementations))

## 3. Load a Local Design and Build the Offline Compiler

The design fixture uses parts that are present in the local CIDAR MoClo collections.

In [ ]:
design_doc = sbol2.Document()
design_doc.read(str(TEST_FILES / "moclo_parts_circuit.xml"))
design = extract_toplevel_definition(design_doc)

compiler = BuildCompiler.from_local_documents(collection_docs, design_doc=design_doc)

print("Design:", design.displayId, "components=", len(design.components))
print("Merged document component definitions:", len(compiler.sbol_doc.componentDefinitions))
print("Indexed plasmids:", len(compiler.indexed_plasmids))
print("Indexed backbones:", len(compiler.indexed_backbones))
print("Restriction enzyme implementations:", len(compiler.restriction_enzyme_implementations))
print("Ligase implementations:", len(compiler.ligase_implementations))

## 4. Run Level-1 Assembly Offline

This runs the level-1 MoClo assembly using only local SBOL data. The output is an SBOL document containing the assembly product and provenance activity.

In [ ]:
product_doc = sbol2.Document()
assembly_dict, product_doc = compiler.assembly_lvl1(
    [design],
    final_doc=product_doc,
    product_name="offline_lvl1",
)
products = assembly_dict[design.identity]

print("Product count:", len(products))
for product in products:
    print("-", product.plasmid_definition.displayId, product.plasmid_definition.identity)

print("Product document component definitions:", len(product_doc.componentDefinitions))
print("Product document implementations:", len(product_doc.implementations))
print("Product document activities:", len(product_doc.activities))

## 5. Write Explicit Outputs

BuildCompiler now avoids writing hidden `output.json` files when no output path is provided. This cell writes explicit notebook outputs under `notebooks/results/buildcompiler_offline_quickstart/`.

In [ ]:
product_sbol_path = RESULTS_DIR / "offline_lvl1_products.xml"
assembly_json_path = RESULTS_DIR / "offline_lvl1_pudu_input.json"

product_sbol_path.write_text(product_doc.writeString(), encoding="utf-8")
assembly_json = compiler.last_assembly_pudu_json
write_assembly_pudu_input_json(assembly_json, assembly_json_path)

print("Wrote:", product_sbol_path)
print("Wrote:", assembly_json_path)
print("Assembly JSON entries:", len(assembly_json))

## 6. Run Two Level-1 Designs Offline

This example reuses the same merged local inventory for two design files: `moclo_parts_circuit.xml` and `mocloparts116.xml`. The second file contains individual part definitions plus one composite design, so the helper below selects the `ComponentDefinition` that has subcomponents.

In [ ]:
def load_composite_design(path: Path):
    doc = sbol2.Document()
    doc.read(str(path))
    candidates = [cd for cd in doc.componentDefinitions if len(cd.components) > 1]
    if not candidates:
        raise ValueError(f"No composite design found in {path}")
    return doc, candidates[0]


design_paths = [
    TEST_FILES / "moclo_parts_circuit.xml",
    TEST_FILES / "mocloparts116.xml",
]

multi_design_docs = []
multi_designs = []
for path in design_paths:
    doc, composite_design = load_composite_design(path)
    multi_design_docs.append(doc)
    multi_designs.append(composite_design)
    print(path.name, "->", composite_design.displayId, "components=", len(composite_design.components))

In [ ]:
multi_compiler = BuildCompiler.from_local_documents(collection_docs, design_doc=multi_design_docs[0])
for extra_design_doc in multi_design_docs[1:]:
    multi_compiler.index_document(extra_design_doc)

multi_results = []
multi_assembly_json = []
for index, composite_design in enumerate(multi_designs, start=1):
    multi_product_doc = sbol2.Document()
    assembly_dict, multi_product_doc = multi_compiler.assembly_lvl1(
        [composite_design],
        final_doc=multi_product_doc,
        product_name=f"offline_multi_{index}",
    )
    products = assembly_dict[composite_design.identity]
    sbol_path = RESULTS_DIR / f"offline_multi_{index}_products.xml"
    sbol_path.write_text(multi_product_doc.writeString(), encoding="utf-8")
    assembly_json = multi_compiler.last_assembly_pudu_json
    multi_assembly_json.extend(assembly_json)
    multi_results.append((composite_design.displayId, products, sbol_path, assembly_json))

multi_json_path = RESULTS_DIR / "offline_multi_pudu_input.json"
write_assembly_pudu_input_json(multi_assembly_json, multi_json_path)

for display_id, products, sbol_path, assembly_json in multi_results:
    print(display_id, "products=", [p.plasmid_definition.displayId for p in products])
    print("  SBOL:", sbol_path)
print("Combined JSON:", multi_json_path, "entries=", len(multi_assembly_json))

## 7. Run Independent Domestication Offline

Domestication can also be run as an independent stage. This example takes one promoter, one RBS, one CDS, and one terminator from the local CIDAR MoClo parts collection, bridges BuildCompiler's indexed local plasmids/vectors into the clean inventory API, and writes generated level-0 SBOL products plus sequence artifacts.

In [ ]:
def indexed_backbones_from_compiler(build_compiler):
    backbones = []
    seen = set()
    for indexed in [*build_compiler.indexed_backbones, *build_compiler.indexed_plasmids]:
        fusion_sites = tuple(getattr(indexed, "fusion_sites", ()) or ())
        antibiotic = getattr(indexed, "antibiotic_resistance", None)
        definition = getattr(indexed, "plasmid_definition", None)
        if not definition or not fusion_sites or antibiotic != "Ampicillin":
            continue
        if definition.identity in seen:
            continue
        seen.add(definition.identity)
        backbones.append(
            IndexedBackbone(
                identity=definition.identity,
                display_id=definition.displayId,
                metadata={
                    "fusion_sites": fusion_sites,
                    "antibiotic": antibiotic,
                    "insertion_index": 0,
                },
                sbol_component=definition,
            )
        )
    return backbones


def indexed_reagents_from_compiler(build_compiler):
    reagents = []
    for impl in build_compiler.restriction_enzyme_implementations:
        definition = build_compiler.sbol_doc.find(impl.built)
        reagents.append(
            IndexedReagent(
                identity=definition.identity,
                display_id=definition.displayId,
                name=definition.displayId,
                reagent_type="restriction_enzyme",
            )
        )
    for impl in build_compiler.ligase_implementations:
        definition = build_compiler.sbol_doc.find(impl.built)
        reagents.append(
            IndexedReagent(
                identity=definition.identity,
                display_id=definition.displayId,
                name=definition.displayId,
                reagent_type="ligase",
            )
        )
    return reagents


domestication_inventory = Inventory(
    backbones=indexed_backbones_from_compiler(compiler),
    reagents=indexed_reagents_from_compiler(compiler),
)
domestication_part_ids = ["J23100", "B0034", "E0030_yfp", "B0015"]
domestication_parts = [
    next(cd for cd in compiler.sbol_doc.componentDefinitions if cd.displayId == display_id)
    for display_id in domestication_part_ids
]
domestication_doc = sbol2.Document()
domestication_results = run_domestication(
    domestication_parts,
    inventory=domestication_inventory,
    source_document=compiler.sbol_doc,
    target_document=domestication_doc,
)
assert all(result.status == StageStatus.SUCCESS for result in domestication_results)

print("Domesticated parts:", len(domestication_results))
for result in domestication_results:
    artifact = result.protocol_artifacts["domestication"]
    print("-", result.products[0].display_id, "backbone=", artifact["backbone_identity"])
    print("  fusion sites:", artifact["fusion_site_names"], artifact["fusion_site_sequences"])
    print("  insert length:", len(artifact["generated_insert_sequence"]))

In [ ]:
domestication_sbol_path = RESULTS_DIR / "offline_domestication_products.xml"
domestication_artifacts_path = RESULTS_DIR / "offline_domestication_artifacts.json"
domestication_pudu_path = RESULTS_DIR / "offline_domestication_pudu_input.json"
domestication_artifacts = [result.protocol_artifacts for result in domestication_results]
domestication_pudu_json = domestication_artifacts_to_pudu_json(domestication_artifacts)

domestication_sbol_path.write_text(domestication_doc.writeString(), encoding="utf-8")
domestication_artifacts_path.write_text(
    json.dumps(domestication_artifacts, indent=4) + "\n",
    encoding="utf-8",
)
write_assembly_pudu_input_json(domestication_pudu_json, domestication_pudu_path)

print("Wrote:", domestication_sbol_path)
print("Wrote:", domestication_artifacts_path)
print("Wrote:", domestication_pudu_path)

## 8. Full Build With Missing Level-1 Part

This example starts with an abstract level-1 design where the promoter has no matching source plasmid in inventory. The first level-1 attempt blocks, the executor promotes the missing promoter to domestication, and then level-1 assembly is retried using the generated domesticated promoter product.

In [ ]:
full_build_doc = sbol2.Document()

full_build_roles = {
    "missing_promoter": "http://identifiers.org/so/SO:0000167",
    "available_rbs": "http://identifiers.org/so/SO:0000139",
    "available_cds": "http://identifiers.org/so/SO:0000316",
    "available_terminator": "http://identifiers.org/so/SO:0000141",
}
full_build_parts = []
for display_id, role in full_build_roles.items():
    part = sbol2.ComponentDefinition(display_id)
    part.roles = [role]
    seq = sbol2.Sequence(f"{display_id}_seq")
    seq.elements = {
        "missing_promoter": "TTGACAGCTAGCTCAGTCCTAGGTATAATGCTAGC",
        "available_rbs": "AGAGAAAGAGGAGAAATACTA",
        "available_cds": "ATGAAATAA",
        "available_terminator": "CCAGGCATCAAATAAAACGAAAGGCTCAGTCGAAAGACTGGGCCTTTCGTTTTATCTGTTGTTTGTCGGTGAACGCTCTCTACTAGAGTCACACTGGCTCACCTTCGGGTGGGCCTTTCTGCGTTTATAGCTT",
    }[display_id]
    seq.encoding = sbol2.SBOL_ENCODING_IUPAC
    full_build_doc.addSequence(seq)
    part.sequences = [seq.identity]
    full_build_doc.addComponentDefinition(part)
    full_build_parts.append(part)

full_build_design = sbol2.ComponentDefinition("abstract_lvl1_missing_promoter")
for part in full_build_parts:
    component = full_build_design.components.create(f"{part.displayId}_component")
    component.definition = part.identity
full_build_doc.addComponentDefinition(full_build_design)

dom_backbone = sbol2.ComponentDefinition("full_build_DVA_AB")
dom_backbone_seq = sbol2.Sequence("full_build_DVA_AB_seq")
dom_backbone_seq.elements = "AAAACCCCGGGGTTTT"
dom_backbone_seq.encoding = sbol2.SBOL_ENCODING_IUPAC
full_build_doc.addSequence(dom_backbone_seq)
dom_backbone.sequences = [dom_backbone_seq.identity]
full_build_doc.addComponentDefinition(dom_backbone)

available_part_plasmids = [
    IndexedPlasmid(
        identity=f"https://example.org/plasmids/{part.displayId}",
        display_id=f"{part.displayId}_plasmid",
        metadata={"insert_identities": [part.identity]},
    )
    for part in full_build_parts
    if part.displayId != "missing_promoter"
]

full_build_inventory = Inventory(
    plasmids=available_part_plasmids,
    backbones=[
        IndexedBackbone(
            identity=dom_backbone.identity,
            display_id=dom_backbone.displayId,
            metadata={"fusion_sites": ("A", "B"), "antibiotic": "Ampicillin", "insertion_index": 0},
            sbol_component=dom_backbone,
        ),
        IndexedBackbone(
            identity="https://example.org/backbones/lvl1_acceptor",
            display_id="lvl1_acceptor",
            metadata={"stage": BuildStage.ASSEMBLY_LVL1.value},
        ),
    ],
    reagents=[
        IndexedReagent("https://example.org/reagents/BsaI", name="BsaI", reagent_type="restriction_enzyme"),
        IndexedReagent("https://example.org/reagents/T4_DNA_ligase", name="T4_DNA_ligase", reagent_type="ligase"),
    ],
)

class NotebookAssemblyService:
    def run(self, job):
        product = sbol2.ComponentDefinition(f"{job.product_display_id}_assembled")
        job.target_document.addComponentDefinition(product)
        return AssemblySbolResult(
            products=[
                IndexedPlasmid(
                    identity=product.identity,
                    display_id=product.displayId,
                    state=MaterialState.GENERATED,
                    metadata={
                        "insert_identities": [job.product_identity],
                        "selected_part_plasmids": [plasmid.identity for plasmid in job.part_plasmids],
                        "backbone_identity": job.backbone.identity,
                    },
                    sbol_component=product,
                )
            ],
            stage_document=job.target_document,
            activity_identity="notebook_lvl1_activity",
            logs=["Notebook assembly service produced lvl1 product."],
        )

class NotebookPlanner:
    def plan(self, abstract_designs, *, options=None):
        ordered_part_ids = [part.identity for part in full_build_parts]
        return BuildPlan(
            lvl1_requests=[
                BuildRequest(
                    id="notebook:assembly_lvl1:missing_promoter",
                    stage=BuildStage.ASSEMBLY_LVL1,
                    source_identity=full_build_design.identity,
                    source_display_id=full_build_design.displayId,
                    source_kind=DesignKind.COMPONENT_DEFINITION,
                    constraints={"ordered_part_identities": ordered_part_ids},
                )
            ]
        )

full_build_options = BuildOptions()
full_build_executor = FullBuildExecutor.from_dependencies(
    inventory=full_build_inventory,
    sbol_document=full_build_doc,
    options=full_build_options,
    lvl1_stage=AssemblyLvl1Stage(
        inventory=full_build_inventory,
        assembly_service=NotebookAssemblyService(),
        options=full_build_options,
    ),
)
full_build_compiler = OrchestratedBuildCompiler(
    inventory=full_build_inventory,
    sbol_document=full_build_doc,
    planner=NotebookPlanner(),
    executor=full_build_executor,
    options=full_build_options,
)

full_build_result = full_build_compiler.full_build([full_build_design])
print("Full build status:", full_build_result.status.value)
print("Stage path:", [result.stage.value + ':' + result.status.value for result in full_build_result.stage_results])
print("Final products:", [product.display_id for product in full_build_result.final_products])

In [ ]:
full_build_summary_path = RESULTS_DIR / "offline_full_build_missing_part_summary.json"
full_build_summary = {
    "status": full_build_result.status.value,
    "stage_path": [
        {"stage": result.stage.value, "status": result.status.value, "logs": result.logs}
        for result in full_build_result.stage_results
    ],
    "final_products": [product.identity for product in full_build_result.final_products],
    "domesticated_products": [
        product.identity
        for result in full_build_result.stage_results
        if result.stage == BuildStage.DOMESTICATION
        for product in result.products
    ],
}
full_build_summary_path.write_text(json.dumps(full_build_summary, indent=4) + "\n", encoding="utf-8")
print("Wrote:", full_build_summary_path)

## Rationale

- `BuildCompiler.from_local_documents(...)` is used so this tutorial does not depend on SynBioHub availability or credentials.
- The four local collection XML files are merged into one SBOL document before indexing so references across parts, plasmids, enzymes, and implementations resolve offline.
- Outputs are written only when explicit paths are supplied, which keeps notebook runs deterministic and avoids hidden files in the working directory.
- The multi-design example writes one combined PUDU input JSON because one robot assembly request should contain all assemblies that are run together.
- The domestication example uses the independent stage API, but its inventory is built from BuildCompiler's indexed local plasmids/vectors rather than synthetic backbone records.
- This notebook uses the current legacy level-1 assembly implementation because it is the path backed by the downloaded CIDAR MoClo SBOL collections. The newer clean assembly API can be layered into a later tutorial once the inventory bridge is complete.